# DubbingStory — Kaggle Local Vision (Qwen3-VL)

Panduan ini menjelaskan cara menjalankan pipeline DubbingStory di cloud GPU (Kaggle) menggunakan model vision HuggingFace secara lokal, sehingga Anda bisa menganalisa video **tanpa biaya API model vision**!

Kita akan menggunakan **Qwen3-VL-2B-Instruct** (via `vLLM`), model vision open-source yang sangat kompeten dan muat di GPU T4 gratisan yang disediakan Kaggle.

### Persiapan Environment
Pastikan Anda telah mengaktifkan **GPU Accelerator** (T4 x2) di kanan menu pengaturan Kaggle Notebook.

## 1. Clone Repo & Install Dependencies

In [ ]:
# Clone repository langsung ke current directory agar tidak nested
!rm -rf ./* ./.*
!git clone -b main https://github.com/NaufalRizqullah/dubbingstory.git .

# Install dependencies via requirements.txt
# (tambahkan openai dan vllm karena dibutuhkan untuk Local Vision Server)
!pip install -r requirements.txt openai vllm

## 2. Setup API Key via Kaggle Secrets

DubbingStory menggunakan Gemini API **hanya untuk merangkai narasi (script writer)**. Model *vision* akan dijalankan secara lokal.

1. Buka tab **Secrets** (kunci) di panel kiri Kaggle.
2. Tambahkan rahasia baru dengan nama `GOOGLE_API_KEY` dan isikan API Key Gemini Anda.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path

API_KEY_GEMINI = UserSecretsClient().get_secret('GOOGLE_API_KEY') or ""

# Create .env File
env_text = f"""# Auto-generated from notebook userdata
GOOGLE_API_KEY={API_KEY_GEMINI}
"""

Path(".env").write_text(env_text, encoding="utf-8")
print("File .env berhasil dibuat")

## 3. Jalankan Pipeline (vLLM + DubbingStory)

Cell ini akan secara otomatis:
1. Menjalankan server `vLLM` di background (download model `Qwen3-VL-2B-Instruct`).
2. Menunggu server siap merespon.
3. Menjalankan pipeline DubbingStory penuh (menggunakan **Piper TTS** agar TTS juga berjalan offline).
4. Mematikan server vLLM saat selesai.

In [ ]:
import subprocess
import time
import urllib.request
import json
import sys
import os
from dotenv import load_dotenv
load_dotenv()

# --- SETTINGS ---
VIDEO_INPUT = "https://www.youtube.com/watch?v=7glhmGv9mHk" # Bisa URL atau path file lokal
PROJECT_NAME = "my_dubbing_project"
STYLE = "viral_fb"
LANGUAGE = "id"
RATIO = "16:9"

# --- VISION MODEL ---
MODEL_NAME = "Qwen/Qwen3-VL-2B-Instruct"
PORT = 8000
BASE_URL = f"http://127.0.0.1:{PORT}/v1"

def wait_for_server(url, timeout=300):
    print(f"\n⏳ Waiting for vLLM server to start at {url} (timeout: {timeout}s)...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            req = urllib.request.Request(f"{url}/models")
            with urllib.request.urlopen(req) as response:
                if response.status == 200:
                    data = json.loads(response.read().decode())
                    print(f"\n✅ vLLM Server is ready! Available models: {[m['id'] for m in data['data']]}")
                    return True
        except Exception:
            pass
        sys.stdout.write(".")
        sys.stdout.flush()
        time.sleep(5)
    print("\n❌ Timeout waiting for server.")
    return False

if not os.environ.get("GOOGLE_API_KEY"):
    print("❌ ERROR: GOOGLE_API_KEY belum di-set di cell sebelumnya (atau di Kaggle Secrets)!")
else:
    print(f"🚀 Starting vLLM server with model: {MODEL_NAME}...")
    vllm_cmd = [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL_NAME,
        "--port", str(PORT),
        "--max-model-len", "4096", 
        "--enforce-eager"
    ]
    
    vllm_log = open("vllm_server.log", "w")
    vllm_process = subprocess.Popen(vllm_cmd, stdout=vllm_log, stderr=subprocess.STDOUT)
    
    try:
        if wait_for_server(BASE_URL):
            print("\n🚀 Starting DubbingStory Pipeline...")
            
            cmd = [
                sys.executable, "main.py", "run",
                "--input", VIDEO_INPUT,
                "--project", PROJECT_NAME,
                "--style", STYLE,
                "--lang", LANGUAGE,
                "--ratio", RATIO,
                "--vision-provider", "openai",
                "--vision-model", MODEL_NAME,
                "--vision-base-url", BASE_URL,
                "--engine", "piper"
            ]
            
            if "http" in VIDEO_INPUT:
                cmd[cmd.index("--input")] = "--url"
                cmd.append("--i-have-rights")
                
            print(f"Executing: {' '.join(cmd)}")
            subprocess.run(cmd, check=True)
            print("\n🎉 Pipeline completed successfully!")
            print(f"📂 Check the 'outputs/{PROJECT_NAME}/' directory for your dubbed video.")
        else:
            print("Failed to start vision server. Check vllm_server.log for details.")
    except Exception as e:
        print(f"\n❌ An error occurred: {e}")
    finally:
        print("\n🛑 Shutting down vLLM server...")
        vllm_process.terminate()
        vllm_process.wait()
        vllm_log.close()
